# 1) Errors Extracted From Previous Notebook

## Errors Detected During EDA - Full Catalog

| # | Category | Column(s) | Description | Classification | Severity | Affected Rows | % | Examples | Impact | Remediation |
|---|----------|-----------|-------------|----------------|----------|---------------|-----|----------|--------|-------------|
| 1 | Format | name | Trailing whitespace in listing names | Actual Error | Medium | 236 | 0.48% | `'Charming West Village Apartment '` | Join mismatches, duplicate grouping | Auto-fix: trim whitespace |
| 2 | Format | name | Inconsistent capitalization in listing names | Business Anomaly | Low | 377 groups | — | `'Home Away From Home'` / `'HOME AWAY FROM HOME'` (6 variations) | Inconsistent grouping in reports | Flag for review: apply title case standardization |
| 3 | Format | host_name | Inconsistent capitalization in host names | Actual Error | Low | 24 groups | 0.05% | `MIchele` vs `Michele`, `FLora` vs `Flora` | Minor, mostly typos | Auto-fix: likely typos, flag for manual review |
| 4 | Format | name | Non-standard null representation (`'.'`) | Actual Error | Low | 1 | 0.002% | `'.'` | Counted as valid data instead of null | Auto-fix: replace with null |
| 5 | Format | name | Embedded newline characters in listing names | Actual Error | Medium | 168 | 0.34% | `'Cozy private room in NYC \n25 mins from Midtown'` | Parsing failures, export corruption | Auto-fix: remove/replace `\n` |
| 6 | Format | name | Embedded multi whitespaces in listing names | Actual Error | Low | 3+ | <0.01% | `'Huge 2 BR Upper East  Cental Park'` | Inconsistent text formatting, search/filter impact | Auto-fix: collapse multiple spaces |
| 7 | Accuracy | reviews_per_month | Range violation (>30 reviews/month max) | Actual Error | Low | 1 | 0.002% | `58.5` | Unrealistic review velocity | Auto-fix: flag for review, possible data error |
| 8 | Completeness | last_review, reviews_per_month | 20.56% missing values (coupled) | Edge Case | High | 10,052 | 20.56% | NULL when number_of_reviews=0 | Incomplete data but logically consistent | Accept as-is: MNAR pattern, expected when no reviews |
| 9 | Completeness | multiple | 15 rows with 3 missing values | Actual Error | Medium | 15 | 0.03% | — | Possible data entry failures | Flag for review: investigate these 15 rows |
| 10 | Completeness | last_review, reviews_per_month | MNAR pattern confirmed (missing when reviews=0) | Edge Case | Low | 10,052 | 20.56% | Varies by borough: Manhattan 23.22% | Expected behavior, document for context | Accept as-is: logically consistent missingness |
| 11 | Completeness | name, host_name | Isolated missing values, not correlated (MCAR) | Actual Error | Low | 16 (name), 21 (host_name) | 0.03–0.04% | Random missing names | Minor gaps, no correlation between columns | Flag for review: small volume, low priority |
| 12 | Consistency | minimum_nights, availability_365 | Min nights exceeds available days | Business Anomaly | High | 833 | 1.70% | min_nights=30, availability=9 days | Impossible bookings | Flag for review: may be valid for long-term rentals with limited calendar |
| 13 | Consistency | price, availability_365 | Available listings with $0 price | Business Anomaly | Medium | 8 | 0.02% | price=0, availability=28–333 days | Revenue reporting errors | Flag for review: possible data entry error or promotional listings |
| 14 | Consistency | price, room_type | Entire home/apt listed for $0 | Actual Error | High | 2 | 0.004% | Entire home/apt, price=0 | Broken business logic | Auto-fix: likely error, flag for correction |
| 15 | Consistency | price, availability_365 | Unavailable listings still priced | Edge Case | Low | 17,530 | 35.85% | price=$80, availability=0 | Inactive listings with residual pricing | Accept as-is: common for inactive listings, low impact |
| 16 | Consistency | reviews_per_month, number_of_reviews | Monthly rate disproportionate to total reviews | Business Anomaly | Medium | 6,412 | 13.11% | 58.5/month but only 156 total | May indicate recently active listings, not necessarily wrong | Flag for review: could be new highly popular listings |
| 17 | Consistency | host_id, host_name | Host IDs with no host name | Actual Error | Medium | 21 | 0.04% | host_id=526653, host_name=NULL | Broken host identification | Auto-fix: flag for data completion |
| 18 | Accuracy | price | Price outliers (>$873, 3σ above mean) | Business Anomaly | Medium | 388 | 0.79% | $10,000, $9,999, $8,500 | Skews averages, may be luxury valid | Flag for review: luxury listings may be legitimate |
| 19 | Accuracy | minimum_nights | Minimum nights outliers (>68, 3σ above mean) | Business Anomaly | Medium | 327 | 0.67% | 1,250, 1,000, 999 nights | Skews averages, may be long-term valid | Flag for review: long-term rentals may be legitimate |
| 20 | Accuracy | number_of_reviews | Review count outliers (>157, 3σ above mean) | Edge Case | Low | 1,221 | 2.50% | 629, 607, 597 reviews | Highly popular listings, not errors | Accept as-is: legitimate popular listings |
| 21 | Accuracy | reviews_per_month | Review rate outliers (>6.41/month, 3σ above mean) | Business Anomaly | Medium | 608 | 1.24% | 58.5, 27.95, 20.94/month | Unusually high review velocity | Flag for review: possible review manipulation or viral listings |
| 22 | Accuracy | calculated_host_listings_count | Host listing count outliers (>106, 3σ above mean) | Business Anomaly | Medium | 680 | 1.39% | 327 listings (multiple hosts) | Commercial operators vs individual hosts | Flag for review: legitimate property managers |
| 23 | Accuracy | price, minimum_nights, number_of_reviews | Severely skewed distributions (skew >2, kurtosis >7) | Edge Case | Low | All | 100% | price skew=19.12, min_nights skew=21.83 | Affects statistical modeling assumptions | Accept as-is: expected for marketplace data |
| 24 | Timeliness | N/A | Missing load/ingestion timestamp column | Actual Error | Medium | All | 100% | No ingestion_timestamp column | Cannot track data freshness or pipeline latency | Auto-fix: add ingestion_timestamp to Bronze layer |
| 25 | Timeliness | last_review | Temporal gaps in review activity (9 months) | Edge Case | Low | 9 months | — | 2011-06 through 2013-02 gaps | Minor gaps in early sparse data | Accept as-is: low review volume in early years |

---

## Stakeholder Escalation Required

| Issue # | Question for Stakeholder |
|---------|-------------------------|
| 2 | Should listing names be standardized to title case? |
| 12 | Are listings with min_nights > availability valid (long-term rentals with limited calendar)? |
| 13 | Are $0 listings promotional or data entry errors? |
| 18 | What is the maximum valid price per room type? Should luxury outliers be capped? |
| 19 | What is the maximum valid minimum_nights? Are 1,250-night listings valid? |
| 21 | Is unusually high reviews_per_month a concern (review manipulation)? |

---

## Summary by Classification

| Classification | Count | Action |
|----------------|-------|--------|
| Actual Error | 12 | Auto-fix or flag for correction |
| Business Anomaly | 8 | Escalate to stakeholder |
| Edge Case | 7 | Accept as-is, document |


# 2) Actual Errors

### Actual Errors — Auto-Fix or Flag for Correction

| # | Column(s) | Description | Severity | Affected Rows | Remediation |
|---|-----------|-------------|----------|---------------|-------------|
| 1 | name | Trailing whitespace in listing names | Medium | 236 | Auto-fix: trim whitespace |
| 2 | name | Inconsistent capitalization in listing names | Low | 377 groups | Flag for review: apply title case standardization |
| 3 | host_name | Inconsistent capitalization (typos: `MIchele`, `FLora`) | Low | 24 groups | Auto-fix: correct obvious typos |
| 4 | name | Non-standard null representation (`'.'`) | Low | 1 | Auto-fix: replace with null |
| 5 | name | Embedded newline characters (`\n`) | Medium | 168 | Auto-fix: remove/replace `\n` |
| 6 | name | Embedded multi whitespaces | Low | 3+ | Auto-fix: collapse multiple spaces |
| 7 | reviews_per_month | Range violation (>30 reviews/month) | Low | 1 | Auto-fix: flag for review |
| 9 | multiple | 15 rows with 3 missing values | Medium | 15 | Flag for review: investigate rows |
| 11 | name, host_name | Isolated missing values (MCAR) | Low | 37 total | Flag for review: low priority |
| 14 | price, room_type | Entire home/apt listed for $0 | High | 2 | Auto-fix: flag for correction |
| 17 | host_id, host_name | Host IDs with no host name | Medium | 21 | Auto-fix: flag for data completion |
| 24 | N/A | Missing load/ingestion timestamp column | Medium | All | Auto-fix: add `ingestion_timestamp` to Bronze layer |


## Errors we will Solve

### Actual Errors — Auto-Fix

| # | Column(s) | Description | Severity | Affected Rows | Remediation |
|---|-----------|-------------|----------|---------------|-------------|
| 1 | name | Trailing whitespace in listing names | Medium | 236 | Auto-fix: trim whitespace |
| 2 | name | Inconsistent capitalization in listing names | Low | 377 groups | Flag for review: apply title case standardization |
| 3 | host_name | Inconsistent capitalization (typos: `MIchele`, `FLora`) | Low | 24 groups | Auto-fix: correct obvious typos |
| 4 | name | Non-standard null representation (`'.'`) | Low | 1 | Auto-fix: replace with null |
| 5 | name | Embedded newline characters (`\n`) | Medium | 168 | Auto-fix: remove/replace `\n` |
| 6 | name | Embedded multi whitespaces | Low | 3+ | Auto-fix: collapse multiple spaces |

# 3) Reading the Data

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, length, when, lower

# Initialize Spark and load data
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("AB_NYC_Cleaning").getOrCreate()

df = spark.table("Bronze.AB_NYC_2019")
print(f"✅ Data loaded successfully from Bronze")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 12, Finished, Available, Finished, False)

✅ Data loaded successfully from Bronze


# 4) Implementing Data Cleaning Steps

## Fixing Error #1

In [11]:
# Fix Error #1: Trim trailing whitespace in name column

# Show before
print("BEFORE:")
display(df.select("id", "name").filter(length(col("name")) != length(trim(col("name")))).limit(5))

# Apply fix
df = df.withColumn("name", trim(col("name")))

# Show after - verify same rows now cleaned
print("AFTER:")
display(df.select("id", "name").filter(col("id").isin(2539, 3831, 5022)))

# Verify no remaining issues
remaining = df.select(col("name")).filter(length(col("name")) != length(trim(col("name")))).count()
print(f"Remaining whitespace issues: {remaining}")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 13, Finished, Available, Finished, False)

BEFORE:


SynapseWidget(Synapse.DataFrame, 44147aa8-c55b-41df-baaa-9038509d262d)

AFTER:


SynapseWidget(Synapse.DataFrame, 174405e9-77b3-4185-a32c-a84d201e4f66)

Remaining whitespace issues: 0


## Fixing Error #2

In [12]:
# Fix Error #2: Standardize inconsistent capitalization in name column
from pyspark.sql.functions import lower, col, collect_set, count, size, initcap

# Show before - sample of inconsistent capitalization groups
print("BEFORE - Sample of inconsistent capitalization:")
df.groupBy(lower(col("name")).alias("name_lower")) \
  .agg(collect_set("name").alias("variations"), count("name").alias("count")) \
  .filter(size(col("variations")) > 1) \
  .orderBy(col("count").desc()) \
  .show(5, truncate=False)

# Apply fix: standardize to title case
df = df.withColumn("name", initcap(col("name")))

# Show after - verify same groups now standardized
print("AFTER - Verify previously inconsistent groups:")
df.groupBy(lower(col("name")).alias("name_lower")) \
  .agg(collect_set("name").alias("variations"), count("name").alias("count")) \
  .filter(size(col("variations")) > 1) \
  .orderBy(col("count").desc()) \
  .show(5, truncate=False)

# Verify no remaining inconsistent capitalization groups
remaining_groups = df.groupBy(lower(col("name")).alias("name_lower")) \
  .agg(collect_set("name").alias("variations"), count("name").alias("count")) \
  .filter(size(col("variations")) > 1) \
  .count()
print(f"Remaining inconsistent capitalization groups: {remaining_groups}")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 14, Finished, Available, Finished, False)

BEFORE - Sample of inconsistent capitalization:
+----------------------------+------------------------------------------------------------------------------------------------------------------------------+-----+
|name_lower                  |variations                                                                                                                    |count|
+----------------------------+------------------------------------------------------------------------------------------------------------------------------+-----+
|home away from home         |[HOME AWAY FROM HOME, Home Away From Home, Home away from Home, Home Away from home, Home Away from Home, Home away from home]|33   |
|private room                |[Private Room, Private room, private room]                                                                                    |24   |
|private room in williamsburg|[Private Room in Williamsburg, Private room in Williamsburg, Private Room in WILLIAMSBURG]            

## Fixing Error #3

In [13]:
# First: Extract all case-inconsistent host_name pairs
from pyspark.sql.functions import lower

# Show before - all groups with inconsistent capitalization
print("BEFORE - All inconsistent host_name capitalization groups:")
distinct_names = df.select("host_name").filter(col("host_name").isNotNull()).distinct().rdd.flatMap(lambda x: x).collect()

case_map = {}
for val in distinct_names:
    key = val.lower().strip()
    if key not in case_map:
        case_map[key] = set()
    case_map[key].add(val)

inconsistencies = {k: v for k, v in case_map.items() if len(v) > 1}

for key, variations in inconsistencies.items():
    print(f"  '{key}': {variations}")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 15, Finished, Available, Finished, False)

BEFORE - All inconsistent host_name capitalization groups:
  'roseanne': {'Roseanne', 'RoseAnne'}
  'annmarie': {'Annmarie', 'AnnMarie'}
  'lulu': {'Lulu', 'LuLu'}
  'michele': {'MIchele', 'Michele'}
  'mackenzie': {'Mackenzie', 'MacKenzie'}
  'tari': {'TaRi', 'Tari'}
  'kimberly': {'KImberly', 'Kimberly'}
  'rosemarie': {'RoseMarie', 'Rosemarie'}
  'weiwei': {'WeiWei', 'Weiwei'}
  'flora': {'Flora', 'FLora'}
  'jacqueline': {'JacQueline', 'Jacqueline'}
  'latoya': {'LaToya', 'Latoya'}
  'jojo': {'JoJo', 'Jojo'}
  'jolynn': {'JoLynn', 'Jolynn'}
  'joann': {'Joann', 'JoAnn'}
  'janae': {'Janae', 'JaNae'}
  'matt': {'Matt', 'MaTT'}
  'deshawn': {'Deshawn', 'DeShawn'}
  'emily': {'Emily', 'EmiLy'}
  'deanna': {'Deanna', 'DeAnna'}
  'latasha': {'LaTasha', 'Latasha'}
  'deedee': {'DeeDee', 'Deedee'}
  'estelle': {'Estelle', 'EStelle'}
  'mckenzie': {'McKenzie', 'Mckenzie'}


In [14]:
# Fix Error #3: Capitalization typos in host_name
from pyspark.sql.functions import when, col

# Show before
print("BEFORE:")
display(df.select("host_name").filter(
    col("host_name").isin("MIchele", "FLora", "KImberly", "TaRi", "RoseAnne", "AnnMarie", "LuLu", "MacKenzie", "RoseMarie", "WeiWei", "JacQueline", "LaToya", "JoJo", "JoLynn", "JoAnn", "JaNae", "MaTT", "DeShawn", "EmiLy", "DeAnna", "LaTasha", "DeeDee", "EStelle", "McKenzie")
).distinct())

# Apply fix - normalize to proper case (only first letter of each word capitalized)
df = df.withColumn("host_name",
    when(col("host_name") == "MIchele", "Michele")
    .when(col("host_name") == "FLora", "Flora")
    .when(col("host_name") == "KImberly", "Kimberly")
    .when(col("host_name") == "TaRi", "Tari")
    .when(col("host_name") == "RoseAnne", "Roseanne")
    .when(col("host_name") == "AnnMarie", "Annmarie")
    .when(col("host_name") == "LuLu", "Lulu")
    .when(col("host_name") == "MacKenzie", "Mackenzie")
    .when(col("host_name") == "RoseMarie", "Rosemarie")
    .when(col("host_name") == "WeiWei", "Weiwei")
    .when(col("host_name") == "JacQueline", "Jacqueline")
    .when(col("host_name") == "LaToya", "Latoya")
    .when(col("host_name") == "JoJo", "Jojo")
    .when(col("host_name") == "JoLynn", "Jolynn")
    .when(col("host_name") == "JoAnn", "Joann")
    .when(col("host_name") == "JaNae", "Janae")
    .when(col("host_name") == "MaTT", "Matt")
    .when(col("host_name") == "DeShawn", "Deshawn")
    .when(col("host_name") == "EmiLy", "Emily")
    .when(col("host_name") == "DeAnna", "Deanna")
    .when(col("host_name") == "LaTasha", "Latasha")
    .when(col("host_name") == "DeeDee", "Deedee")
    .when(col("host_name") == "EStelle", "Estelle")
    .when(col("host_name") == "McKenzie", "Mckenzie")
    .otherwise(col("host_name"))
)

# Show after
print("AFTER:")
display(df.select("host_name").filter(
    col("host_name").isin("Michele", "Flora", "Kimberly", "Tari", "Roseanne", "Annmarie", "Lulu", "Mackenzie", "Rosemarie", "Weiwei", "Jacqueline", "Latoya", "Jojo", "Jolynn", "Joann", "Janae", "Matt", "Deshawn", "Emily", "Deanna", "Latasha", "Deedee", "Estelle", "Mckenzie")
).distinct())

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 16, Finished, Available, Finished, False)

BEFORE:


SynapseWidget(Synapse.DataFrame, 8a897db1-4dd6-4e6b-b15d-83c4d2d3bbf4)

AFTER:


SynapseWidget(Synapse.DataFrame, 69da4ca2-c6da-48b1-897a-198de5685e5c)

## Fixing Error #4

In [15]:
# Fix Error #4: Replace '.' with null in name column
from pyspark.sql.functions import when, col

# Show before
print("BEFORE:")
display(df.select("id", "name").filter(col("name") == "."))

# Apply fix
df = df.withColumn("name", when(col("name") == ".", None).otherwise(col("name")))

# Show after
print("AFTER:")
display(df.select("id", "name").filter(col("id").isin(df.select("id").filter(col("name") == ".").collect()[0][0] if df.filter(col("name") == ".").count() > 0 else None)))

# Verify
remaining = df.filter(col("name") == ".").count()
print(f"Remaining '.' values in name: {remaining}")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 17, Finished, Available, Finished, False)

BEFORE:


SynapseWidget(Synapse.DataFrame, 668cb521-fe22-4a3d-b190-ef72a5e3a98c)

AFTER:


SynapseWidget(Synapse.DataFrame, 1c12392e-7565-4c11-9edb-c133bed06b0d)

Remaining '.' values in name: 0


## Fixing Error #5

In [16]:
# Fix Error #5: Remove embedded newline characters in name column
from pyspark.sql.functions import regexp_replace, col

# Show before
print("BEFORE:")
display(df.select("id", "name").filter(col("name").contains("\n")).limit(5))

# Apply fix - replace \n with space
df = df.withColumn("name", regexp_replace(col("name"), "\n", " "))

# Show after
print("AFTER:")
display(df.select("id", "name").filter(col("id").isin(
    df.select("id").filter(col("name").contains("\n")).limit(5).rdd.flatMap(lambda x: x).collect()
)))

# Verify
remaining = df.filter(col("name").contains("\n")).count()
print(f"Remaining newline characters in name: {remaining}")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 18, Finished, Available, Finished, False)

BEFORE:


SynapseWidget(Synapse.DataFrame, b43448eb-55c3-46ab-88a4-e3d19950f0cc)

AFTER:


SynapseWidget(Synapse.DataFrame, 69bbbeb8-87ec-4e46-90ea-abe684aa5c24)

Remaining newline characters in name: 0


## Fixing Error #6

In [17]:
# Find names with double/multiple spaces
from pyspark.sql.functions import col

df.select("id", "name").filter(col("name").contains("  ")).show(truncate=False)

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 19, Finished, Available, Finished, False)

+------+--------------------------------------------------+
|id    |name                                              |
+------+--------------------------------------------------+
|7750  |Huge 2 Br Upper East  Cental Park                 |
|23686 |2000 Sf 3br 2bath West Village Private  Townhouse |
|27385 |Great Large 1 Br Apt  In East Village!            |
|28321 |Large 1  Br In A 3 Br Brooklyn Apt. Next To Q Trn.|
|32037 |Huge Private  Floor At The Waverly                |
|40453 |Charming & Cozy Midtown Loft Any Week Ends  !!!   |
|46723 |Safe  And Beautiful Accomodation                  |
|50447 |Lovely Apt & Garden;  Legal;  Best Area; Amenities|
|54508 |Sml Rm In Pr Brst  Park Sl Great For Med/students |
|61492 |Exclusive Room With Private Bath In  Les          |
|63657 |Private, Large & Sunny Top Floor Apt  W/w&d       |
|63913 |Hosting Your  Sunny, Spacious Nyc Room            |
|64107 |Brooklyn  Studio Apartment                        |
|64277 |Bedroom2 For Rent  10min From Ma

In [18]:
# Fix Error #5

from pyspark.sql.functions import regexp_replace

df = df.withColumn("name", regexp_replace(col("name"), " +", " "))

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 20, Finished, Available, Finished, False)

In [19]:
# Find names with double/multiple spaces
from pyspark.sql.functions import col

df.select("id", "name").filter(col("name").contains("  ")).show(truncate=False)

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 21, Finished, Available, Finished, False)

+---+----+
|id |name|
+---+----+
+---+----+



# 5) Saving Cleaned Data to Silver Layer

In [20]:
# Save cleaned data to Silver layer
df.write.mode("overwrite").format("delta").saveAsTable("Silver.AB_NYC_2019")

print("✅ Data saved successfully to Silver.AB_NYC_2019")

StatementMeta(, a3cf9fce-8ddf-4686-8af8-4b93a56eaa8a, 22, Finished, Available, Finished, False)

✅ Data saved successfully to Silver.AB_NYC_2019
